In [ ]:
# RSNA Knee - frozen DINOv2-S embedding pass (Kaggle T4, internet OFF).
# Full 4407-study run -> embeddings.npz (checkpointed every 500 studies).
SMOKE = False
K_PER_PLANE = 16      # slices sampled per plane
IMG = 224
import os, glob, time, numpy as np, pandas as pd, torch
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

base = None
for r in sorted(glob.glob('/kaggle/input/*')):
    if os.path.isdir(r) and ('competition' in r.lower() or glob.glob(r+'/**/train_series.csv', recursive=True)):
        hit = glob.glob(r+'/**/train_series.csv', recursive=True)
        if hit: base = os.path.dirname(hit[0]); break
print('COMP base:', base)
ts = pd.read_csv(os.path.join(base, 'train_series.csv'))
tr = pd.read_csv(os.path.join(base, 'train.csv'))
print('series', ts.shape, 'train', tr.shape)

In [ ]:
# ---- load frozen DINOv2-S offline via HuggingFace (config.json + pytorch_model.bin) ----
from transformers import AutoModel
mdir = None
for c in glob.glob('/kaggle/input/**/config.json', recursive=True):
    d = os.path.dirname(c)
    if glob.glob(d + '/pytorch_model.bin') or glob.glob(d + '/*.safetensors'):
        mdir = d; break
print('MODEL DIR:', mdir, '|', os.listdir(mdir) if mdir else None)
model = AutoModel.from_pretrained(mdir).eval().half().cuda()
EMB_DIM = model.config.hidden_size
with torch.no_grad():
    o = model(pixel_values=torch.zeros(2, 3, IMG, IMG).half().cuda(), interpolate_pos_encoding=True)
print('sanity last_hidden', tuple(o.last_hidden_state.shape), '| EMB_DIM', EMB_DIM)

In [ ]:
# ---- DICOM helpers ----
import pydicom, torch.nn.functional as F
IM_MEAN = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1)
IM_STD  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

def series_slice_paths(series_dir):
    fs = glob.glob(series_dir+'/*.dcm')
    hdr = []
    for f in fs:
        try:
            d = pydicom.dcmread(f, stop_before_pixels=True)
            inst = float(getattr(d,'InstanceNumber', 0) or 0)
            hdr.append((inst, f))
        except Exception:
            hdr.append((0.0, f))
    hdr.sort()
    return [f for _,f in hdr]

def sample_idx(n, k):
    if n <= k: return list(range(n))
    lo, hi = int(n*0.12), int(n*0.88)
    if hi<=lo: lo,hi=0,n
    return [int(round(x)) for x in np.linspace(lo, hi-1, k)]

def decode(path):
    d = pydicom.dcmread(path)
    a = d.pixel_array.astype(np.float32)
    a = a*float(getattr(d,'RescaleSlope',1) or 1) + float(getattr(d,'RescaleIntercept',0) or 0)
    return a

def norm_resize(a):
    lo, hi = np.percentile(a, 0.5), np.percentile(a, 99.5)
    a = np.clip((a-lo)/max(hi-lo,1e-6), 0, 1)
    t = torch.from_numpy(a)[None,None].float()
    t = F.interpolate(t, size=(IMG,IMG), mode='bilinear', align_corners=False)[0,0]
    return t

def embed_series(paths):
    idx = sample_idx(len(paths), K_PER_PLANE)
    tris = []
    cache = {}
    def get(i):
        i = max(0, min(len(paths)-1, i))
        if i not in cache:
            try: cache[i] = norm_resize(decode(paths[i]))
            except Exception: cache[i] = torch.zeros(IMG,IMG)
        return cache[i]
    for i in idx:
        tris.append(torch.stack([get(i-1), get(i), get(i+1)]))
    x = torch.stack(tris)                     # [k,3,IMG,IMG]
    x = (x - IM_MEAN)/IM_STD
    with torch.no_grad():
        o = model(pixel_values=x.half().cuda(), interpolate_pos_encoding=True)
        z = o.last_hidden_state[:,0].float().cpu().numpy()   # CLS token [k, EMB_DIM]
    return z
print('helpers ready')

In [ ]:
# ---- embed loop (checkpointed) ----
PLANE_ID = {'Sagittal':0,'Coronal':1,'Axial':2}
studies = list(tr['StudyInstanceUID'])
gold_first = list(tr[tr['ACL'].notna()]['StudyInstanceUID'])
order = gold_first + [s for s in studies if s not in set(gold_first)]
if SMOKE: order = order[:3]
print('embedding', len(order), 'studies')

def save(n_done):
    np.savez_compressed('embeddings.npz',
                        emb=np.array(emb_list, dtype=np.float16),
                        meta=np.array(meta_rows, dtype=np.int32),
                        study_ids=np.array(order[:n_done]))

emb_list, meta_rows = [], []
t0 = time.time()
for si, study in enumerate(order):
    srows = ts[ts['StudyInstanceUID']==study]
    for _, sr in srows.iterrows():
        sdir = os.path.join(base, 'train_series', study, sr['SeriesInstanceUID'])
        if not os.path.isdir(sdir):
            alt = glob.glob(os.path.join(base,'**',sr['SeriesInstanceUID']), recursive=True)
            sdir = alt[0] if alt else None
        if not sdir: continue
        paths = series_slice_paths(sdir)
        if not paths: continue
        try:
            z = embed_series(paths)
        except Exception as e:
            print('  ERR study', si, str(e)[:60]); continue
        pid = PLANE_ID.get(sr['Anatomical_Plane'], 0)
        for j in range(z.shape[0]):
            emb_list.append(z[j]); meta_rows.append((si, pid, int(sr['Fluid_Sensitive'])))
    if si % 200 == 0:
        print(f'  {si}/{len(order)} slices={len(emb_list)} {time.time()-t0:.0f}s', flush=True)
    if si and si % 500 == 0:
        save(si+1); print('  checkpoint saved at', si+1, flush=True)

save(len(order))
emb = np.array(emb_list, dtype=np.float16)
print('DONE emb', emb.shape, 'studies', len(order), f'{time.time()-t0:.0f}s', 'MB', round(os.path.getsize('embeddings.npz')/1e6,1))